# Setup

In [1]:
import numpy as np, timeit, time, matplotlib.pyplot as plt, json, os
from tqdm import tqdm

In [2]:
from binary_models import *
from benchmark_models import SMK_model, get_SMK_dim_labels
from benchmark_models import get_nSMK_SCM, get_nSMK_variables, nSMK_simulation, avg_nSMK_simulation
from evaluation import evaluate_full
from actualcauses import beam_search, show_rules, iterative_identification
# from actualcauses.base_algorithm import *

# Functions

In [3]:
# def beam_search(
#     instance, domains, simulation, variables, # SCM
#     max_steps=5, beam_size=10, epsilon=.05, early_stop=True, max_time=None, # Parameters
#     var_mapping=None, ref_w=tuple(), Cs=None, # Additional parameters when running for sub-instance
#     verbose=0, 
#     ):
#     # verbose: 
#     #  = 1 -> best cause at the end, tqdm for steps
#     #  >= 2 -> removes step tqdm, adds step header + number of cause found + best and worse non causes
#     #  >= 3 -> adds all causes + tqdm for get_rules
    
#     all_causes = []
#     init_time = time.time()
#     if Cs is None: Cs = []
#     if max_steps == -1 or max_steps is None: iterator = count(start=1, step=1)
#     else: iterator = range(1,max_steps+1)

#     for t in tqdm(iterator, disable=(verbose!=1)):
#         # Render the step
#         if verbose >= 2: print(f"{f'Step {t}':=^30}")
            
#         # Create the rules for step t base on the ones from t-1, we use the initial ones if t==1
#         beams = get_rules(beams if t > 1 else None, domains, instance, Cs, verbose >= 3)

#         # Check for early stop
#         if check_early_stop(beams, early_stop, all_causes, max_time, init_time):
#             break
            
#         # Render how many nodes will be evaluated
#         if verbose >= 2: print(f"Evaluating {len(beams)} rules")
            
#         # Evaluate the rules using the simulation 
#         sim_beams = map_beams(beams, var_mapping, ref_w)
#         cf_values = simulation(sim_beams)
        
#         # Build the tuples of rule values
#         causes, non_causes = split_rules(beams, cf_values, instance, epsilon)

#         # Filter causes to keep only minimal ones and save them
#         causes = filter_minimality(causes)
#         all_causes += causes
#         for rule_value in causes:
#             Cs.append(rule_value[3])

#         ### Build next beams
#         beams = get_next_beams(non_causes, beam_size, Cs)
        
#         # Render step output
#         render_step(verbose, causes, non_causes, instance, variables)

#     # Sort final rule set
#     all_causes.sort(key=sort_key)

#     # Render final result
#     if verbose:
#         print(f"----> Found {len(all_causes)} causes.")
#         if all_causes:
#             print(f"{'Overall best rule:':=^30}")
#             show_rule(all_causes[0], variables)
        
#     return all_causes

In [4]:
def var2label(S, variables):
    return {variables[i] for i in S}

In [5]:
# This file is inspired by https://github.com/marcotcr/anchor

def kl_bernoulli(p, q):
    p = min(0.9999999999999999, max(0.0000001, p))
    q = min(0.9999999999999999, max(0.0000001, q))
    return (p * np.log(float(p) / q) + (1 - p) *
            np.log(float(1 - p) / (1 - q)))

def dup_bernoulli(p, level):
    lm = p
    um = min(min(1, p + np.sqrt(level / 2.)), 1)
    qm = (um + lm) / 2.
    if kl_bernoulli(p, qm) > level:
        um = qm
    else:
        lm = qm
    return um

def dlow_bernoulli(p, level):
    um = p
    lm = max(min(1, p - np.sqrt(level / 2.)), 0)
    qm = (um + lm) / 2.
    if kl_bernoulli(p, qm) > level:
        lm = qm
    else:
        um = qm
    return lm

def compute_beta(n_features, t):
    delta = .1
    alpha = 1.1
    k = 405.5
    temp = np.log(k * n_features * (t ** alpha) / delta)
    return temp + np.log(temp)




In [6]:
def lucb(simulation, rules, beam_size, a=.05, beam_eps=.1, cause_eps=.01, non_cause_esp=.01, 
         max_iter=200, verbose=1, batch_size=10, init_batch_size=20, lucb_infos=None):
        
    n_arms = len(rules) # Doing armed bandits with the rules to evaluate
    # For each arm we keep track of the number of samples, the number of success, the upper bound and the lower bound
    positives, scores, lb, means, mean_scores = np.zeros((5, n_arms))
    ub = np.ones(n_arms)
    n_samples = np.zeros(n_arms, dtype=int)
    beta = 0
    
    # Utils function
    def action_arm(arm, init=False):
        bs = init_batch_size if init else batch_size
        for _ in range(bs):
            _, label, score = simulation(rules[arm])
            positives[arm] += label
            scores[arm] += score
        n_samples[arm] += bs
        means[arm] = positives[arm] / n_samples[arm]
        mean_scores[arm] = scores[arm] / n_samples[arm]

    def update_bounds_beam(t):
        bs = beam_size + (means < a).sum()
        sorted_rule_ids = sorted(range(n_arms), key = lambda i: (means[i],mean_scores[i]))
        
        beam_ids = sorted_rule_ids[:bs]
        non_beam_ids = sorted_rule_ids[bs:]
        # print(f"{beam_ids=} / {non_beam_ids=}")
        if not beam_ids or not non_beam_ids: return 0
        for i in beam_ids:
            ub[i] = dup_bernoulli(means[i], beta / n_samples[i])
        for i in non_beam_ids:
            lb[i] = dlow_bernoulli(means[i], beta / n_samples[i])
            
        ut = beam_ids[np.argmax(ub[beam_ids])]
        lt = non_beam_ids[np.argmin(lb[non_beam_ids])]
        B = ub[ut] - lb[lt]

        # ro = 2
        # print(f"Update beam bounds (bs={beam_size}+{bs-beam_size}) -> lb={round(lb[lt],ro)}, ut={round(ub[ut],ro)}, B={round(B,ro)}")
        # print(f"means: {means[sorted_rule_ids].round(ro)}")
        # print(f"iter : {n_samples[sorted_rule_ids]}")
        # print(f"lb:    {lb[sorted_rule_ids].round(ro)}")
        # print(f"ub:    {ub[sorted_rule_ids].round(ro)}")
        # print("-"*30)
        if B >= beam_eps:
            action_arm(ut)
            action_arm(lt)
        return B

    def update_bounds_non_cause(t):
        ids = np.argwhere(means >= a).flatten()
        # print(f"non cause: {ids=}")
        for i in ids:
            lb[i] = dlow_bernoulli(means[i], beta / n_samples[i])
        if not ids.size: return 0
        lt = ids[np.argmin(lb[ids])]
        B = a - lb[lt]
        if B >= non_cause_esp:
            action_arm(lt)
        return B

    def update_bounds_cause(t):
        ids = np.argwhere(means < a).flatten()
        # print(f"cause: {ids=}")
        for i in ids:
            ub[i] = dup_bernoulli(means[i], beta / n_samples[i])
        if not ids.size: return 0
        ut = ids[np.argmax(ub[ids])]
        B = ub[ut] - a
        if B >= cause_eps:
            action_arm(ut)
        return B
    
    # Initialization
    beam_bound = 1
    cause_bound = 1
    non_cause_bound = 1
    for arm in tqdm(range(n_arms), disable=not verbose):
        action_arm(arm, True)
    it = 1
    # Loop
    with tqdm(total=n_arms * max_iter, disable=not verbose) as pbar:
        while n_samples.sum() < n_arms * max_iter:
            # Stop condition
            if beam_bound <= beam_eps and cause_bound <= cause_eps and non_cause_bound <= non_cause_esp: 
                if verbose > 1: 
                    print(f"Success: {beam_bound=:.4f} / {cause_bound=:.4f} / {non_cause_bound=:.4f})")
                break
            if cause_bound <= cause_eps and non_cause_bound <= non_cause_esp and beam_size + (means < a).sum() >= n_arms:
                if verbose > 1:
                    print(f"All rules pass on to next state: {cause_bound=:.4f}, {non_cause_bound=:.4f}")
                break
                
            # Update bounds
            beta = compute_beta(n_arms, it)
            beam_bound = update_bounds_beam(it)
            cause_bound = update_bounds_cause(it)
            non_cause_bound = update_bounds_non_cause(it)
            pbar.n = n_samples.sum()
            pbar.refresh()
            it += 1
        else:
            # Render how much we fail if we fail to reach the bound
            if verbose > 1: 
                print(f"Fail: {beam_bound=:.4f} / {cause_bound=:.4f} / {non_cause_bound=:.4f}")
            
    if verbose > 2:
        print(f"ub={ub.round(4)}")
        print(f"lb={lb.round(4)}")
        print(f"preds={means.round(2)}")
        print(f"n_samples={n_samples}")
    if lucb_infos is not None:
        lucb_infos.append({
            "n_calls": int(n_samples.sum())
        })
    outputs = [((n_sample, ub_i, lb_i), mean, mean_score) for \
                   n_sample, ub_i, lb_i, mean, mean_score in \
                    zip(
                        n_samples.tolist(),
                        ub.tolist(), 
                        lb.tolist(), 
                        means.tolist(), 
                        mean_scores.tolist()
                    )]
    return outputs

In [7]:
def lucb_nSMK_simulation(rules, u, n_attacker, N, lucb_params, nl=0):
    sim = lambda rule: nSMK_simulation(rule, u, n_attacker, nl)
    return lucb(sim, rules, **lucb_params)

In [8]:
def get_nSMK_SCM(n_attacker, u, do_lucb=True, N=100, nl=.05, lucb_params={}):
    variables = get_nSMK_variables(n_attacker)
    s, _, _ = nSMK_simulation([], u, n_attacker, nl=0) # No noise for the instance
    domains = ((0,1),)*(len(variables)-1)
    if do_lucb:
        simulation = lambda rules: lucb_nSMK_simulation(rules, u, n_attacker, N, lucb_params, nl)
    else: 
        simulation = lambda rules: avg_nSMK_simulation(rules, u, n_attacker, N, nl)
    return {"variables": variables[:-1], "domains":domains,
            "simulation": simulation, "instance":s}

In [9]:
def compute_comparison(rules, avg_scores, lucb_scores, full_verbose=False):
    count = 0
    for rule, avg_s, lucb_s in zip(rules, avg_scores, lucb_scores):
        is_in_cause = any([rule[0][0] in e[3] for e in sbs_causes])
        order_correct = is_in_cause == (avg_s>lucb_s)
        if full_verbose: print(variables[rule[0][0]], avg_s, round(lucb_s,2), order_correct)
        count += order_correct
    print("Proportion of correct order:", round(count/len(rules),2))

In [10]:
def make_one_lucb(rules, seed=1):
    np.random.seed(seed)
    out = lucb(simulation, rules, **lucb_params)
    lucb_scores = [elt[1] for elt in out]
    
    avg_out = np.array([[[simulation(rule)[1],simulation(rule)[2]] for _ in range(N)] for rule in rules])
    avg_scores = avg_out[:,:,0].mean(1)
    
    compute_comparison(rules, avg_scores, lucb_scores, full_verbose=False)
    
    sorted_ids = sorted(range(len(rules)), key=lambda i: [out[i][1], out[i][2]])
    beam = [rules[i] for i in sorted_ids[:bs]]
    next_rules = get_rules(beam, SCM_lucb["domains"], SCM_lucb["instance"], [])

    found_vars = {variables[i] for i in sorted_ids[:bs]}
    exp_vars = {variables[i] for i in set().union(*[elt[3] for elt in sbs_causes])}
    # print("Found:", found_vars)
    # print("Exp:", exp_vars)
    # print("Correct:", found_vars & exp_vars)
    # print("Missed:", exp_vars - found_vars)
    # print("Added:", found_vars - exp_vars)
    found_naive = {variables[i] for i in np.argsort(avg_scores)[:bs]}
    print(f"Prop correct: lucb={len(found_vars & exp_vars)/bs:.2f} / naive={len(found_naive & exp_vars)/bs:.2f}")
    
    return out, lucb_scores, avg_scores, next_rules

In [11]:
def get_avg_out(rules, simulation, seed=None):
    if seed is not None: np.random.seed(seed)
    out = []
    for rule in rules:
        line = []
        for _ in range(N):
            sim_out = simulation(rule)
            line.append((sim_out[1],sim_out[2]))
        out.append(line)
    return np.array(out)

In [12]:
def compare_prop_corect(rules, nl, n_seeds=30):
    prop_lucb = []
    prop_naive = []
    simulation = lambda rule: nSMK_simulation(rule, u, n_attacker, nl)
    exp_vars = {variables[i] for i in set().union(*[elt[3] for elt in sbs_causes])}
    
    for seed in tqdm(range(n_seeds)):
        np.random.seed(seed)
        
        out = lucb(simulation, rules, **lucb_params|{"verbose":0})
        sorted_ids = sorted(range(len(rules)), key=lambda i: [out[i][1], out[i][2]])
        found_vars = {variables[i] for i in sorted_ids[:bs]}
        prop_lucb.append(len(found_vars & exp_vars)/bs)
        
        avg_out = get_avg_out(rules, simulation)
        avg_scores = avg_out[:,:,0].mean(1)
        found_naive = {variables[i] for i in np.argsort(avg_scores)[:bs]}
        prop_naive.append(len(found_naive & exp_vars)/bs)

    print(f"Prop lucb: {np.mean(prop_lucb):.2f}±{np.std(prop_lucb):.2f}")  
    print(f"Prop naive: {np.mean(prop_naive):.2f}±{np.std(prop_naive):.2f}")           

In [13]:
# bs = 12
# N = 50
# eps = .35
# nl = 1.5/len(variables)
# lucb_params = {"beam_size":bs, "a":eps, "cause_eps":.01, 
#                "beam_eps":.1, "max_iter":N, "verbose":2, 
#                "batch_size": 5, "init_batch_size": 30,
#                "non_cause_esp":.05}
# compare_prop_corect(rules_1, nl=nl, n_seeds=100)
# for nl in (0.001, .01, .1):
#     print(f"{nl=}")
#     compare_prop_corect(rules_1, nl=nl, n_seeds=100)
#     print("-"*50)

In [14]:
# bs = 12
# N = 50
# eps = .35
# nl = .001
# lucb_params = {"beam_size":bs, "a":eps, "cause_eps":.01, 
#                "beam_eps":.1, "max_iter":N, "verbose":2, 
#                "batch_size":3, "non_cause_esp":.05}

In [15]:
# from actualcauses.base_algorithm import get_initial_rules, get_rules
# from benchmark_models import nSMK_simulation
# simulation = lambda rule: nSMK_simulation(rule, u, n_attacker, nl)

# rules_1 = get_initial_rules(SCM_lucb["instance"], SCM_lucb["domains"])

In [16]:
# for seed in (1,2,3,4,5,6):
#     out_1, lucb_scores_1, avg_scores_1, rules_2 = make_one_lucb(rules_1, seed=seed)
#     print("-"*40)

In [17]:
# np.random.seed(1)
# out_2 = lucb(simulation, rules_2, **lucb_params)
# lucb_scores_2 = [elt[1] for elt in out_2]

# Main

Problem to solve: for 5/10 attackers, LUCB misses way more than naive sampling

In [18]:
prefix = "../"
all_contexts = {n_attacker: 
                np.load(f"../results/contexts/{n_attacker=}.npy")
               for n_attacker in n_attackers}

In [19]:
n_attacker = 5
u = all_contexts[n_attacker][10]

In [20]:
variables = get_SMK_dim_labels(n_attacker)
SCM = make_SCM(variables, u, SMK_model, n_attacker=n_attacker)
dag, init_var_ids = build_DAG(n_attacker, SCM["variables"])

In [21]:
t = time.time()
sbs_causes = iterative_identification(**SCM, dag=dag, 
                                            init_var_ids=init_var_ids,
                                            max_steps=-1, beam_size=-1, 
                                            verbose=0, early_stop=False)
print(f"{(time.time()-t):.3f}s")

5.111s


In [22]:
print(f"Found {len(sbs_causes)} causes")

Found 30 causes


In [23]:
ref_causes = [tuple(elt[3]) for elt in sbs_causes]

In [24]:
for e, o, s, C, W, cf in sbs_causes:
    print(f"C={var2label(C, variables)}, W={var2label(W, variables)}")

C={'SD', 'DK'}, W=set()
C={'SD-U1', 'DK-U2'}, W={'DK-U5', 'SD-U5', 'DK-U4'}
C={'SD-U1', 'DK'}, W={'SD-U5'}
C={'SD', 'DK-U2'}, W={'DK-U5', 'DK-U4'}
C={'KMS-U1', 'GP-U2'}, W={'DK-U5', 'SD-U5', 'DK-U4'}
C={'KMS-U1', 'GK-U2'}, W={'DK-U5', 'SD-U5', 'DK-U4'}
C={'KMS-U1', 'DK-U2'}, W={'DK-U5', 'SD-U5', 'DK-U4'}
C={'SD-U1', 'GP-U2'}, W={'DK-U5', 'SD-U5', 'DK-U4'}
C={'SD-U1', 'GK-U2'}, W={'DK-U5', 'SD-U5', 'DK-U4'}
C={'KMS-U1', 'DK'}, W={'SD-U5'}
C={'SD', 'GP-U2'}, W={'DK-U5', 'DK-U4'}
C={'GK-U2', 'SD'}, W={'DK-U5', 'DK-U4'}
C={'FN-U2', 'FS-U2', 'A-U1'}, W={'DK-U5', 'SD-U5', 'DK-U4'}
C={'FN-U2', 'AD-U1', 'FS-U2'}, W={'DK-U5', 'SD-U5', 'DK-U4'}
C={'GP-U2', 'A-U1'}, W={'DK-U5', 'SD-U5', 'DK-U4'}
C={'AD-U1', 'GP-U2'}, W={'DK-U5', 'SD-U5', 'DK-U4'}
C={'FN-U2', 'KMS-U1', 'FS-U2'}, W={'DK-U5', 'SD-U5', 'DK-U4'}
C={'A-U1', 'FF-U2'}, W={'DK-U5', 'SD-U5', 'DK-U4'}
C={'AD-U1', 'FF-U2'}, W={'DK-U5', 'SD-U5', 'DK-U4'}
C={'GK-U2', 'A-U1'}, W={'DK-U5', 'SD-U5', 'DK-U4'}
C={'GK-U2', 'AD-U1'}, W={'DK-U5', 'SD-

In [68]:
bs = 12
N = 20
eps = .35
nl = 1.5/len(variables)
lucb_params = {"beam_size":bs, "a":eps, "cause_eps":.01, 
               "beam_eps":.7, "max_iter":N, "verbose":2, 
               "batch_size": 2, "init_batch_size": 15,
               "non_cause_esp":.05}

In [69]:
SCM_avg = get_nSMK_SCM(n_attacker, u, do_lucb=False, N=N, nl=nl)
SCM_lucb = get_nSMK_SCM(n_attacker, u, do_lucb=True, N=N, nl=nl, lucb_params=lucb_params)

In [70]:
np.random.seed(0)
output_avg = beam_search(**SCM_avg, max_steps=4, beam_size=bs,early_stop=False,verbose=1, epsilon=eps)
print("\nRESULTS\n")
show_rules(output_avg, SCM_avg["variables"])

100%|██████████| 4/4 [00:07<00:00,  1.95s/it]

----> Found 12 causes.
======Overall best rule:======
C={'SD': '0', 'DK': '0'}, W={}, output=0.0, score=0.452

RESULTS

C={'SD': '0', 'DK': '0'}, W={}, output=0.0, score=0.452
C={'AD-U1': '0', 'DK': '0', 'AD-U5': '0', 'GP-U2': '0'}, W={}, output=0.05, score=0.349
C={'GP-U5': '0', 'A-U1': '0', 'DK': '0'}, W={'SD-U5': '0'}, output=0.05, score=0.381
C={'SD': '0', 'GK-U4': '0', 'GP-U2': '0'}, W={'DK-U5': '0'}, output=0.1, score=0.404
C={'DK': '0', 'A-U1': '0', 'AD-U5': '0', 'GP-U2': '0'}, W={}, output=0.15, score=0.357
C={'FS-U2': '0', 'AD-U1': '0', 'DK': '0'}, W={'SD-U5': '0'}, output=0.15, score=0.390
C={'KMS-U1': '0', 'GP-U5': '0', 'DK': '0'}, W={'SD-U5': '0'}, output=0.15, score=0.406
C={'SD-U1': '0', 'DK': '0', 'AD-U5': '0', 'GP-U2': '0'}, W={}, output=0.2, score=0.391
C={'KMS-U1': '0', 'DK': '0', 'AD-U5': '0', 'GP-U2': '0'}, W={}, output=0.25, score=0.373
C={'SD': '0', 'FF-U2': '0', 'GK-U4': '0'}, W={'DK-U5': '0'}, output=0.25, score=0.394
C={'SD': '0', 'GK-U2': '0', 'FF-U4': '0'}, W

In [71]:
np.random.seed(0)
causes = beam_search(**SCM_lucb, max_steps=4, beam_size=bs, early_stop=False,verbose=2, epsilon=eps)
print("\nRESULTS\n")
show_rules(causes, SCM_lucb["variables"])

============Step 1============
Evaluating 57 rules


100%|██████████| 57/57 [00:00<00:00, 579.71it/s]
1143it [00:00, 8823.03it/s]                           


Fail: beam_bound=0.7557 / cause_bound=0.0000 / non_cause_bound=0.1057
Number of causes found: 0
Number of non-causes remaining: 57
Best non-cause:
C={'FS-U1': '1'}, W={}, output=1.0, score=0.535
Worst non-cause:
C={'SD': '0'}, W={}, output=0.9393939393939394, score=0.480
============Step 2============
Evaluating 1278 rules


100%|██████████| 1278/1278 [00:01<00:00, 730.40it/s]
25564it [00:22, 1135.84it/s]                              


Fail: beam_bound=0.8838 / cause_bound=0.0070 / non_cause_bound=0.2338
Number of causes found: 1
Number of non-causes remaining: 1277
Best non-cause:
C={'FF-U3': '0'}, W={'FS-U1': '0'}, output=1.0, score=0.482
Worst non-cause:
C={'SD': '0'}, W={'DK': '1'}, output=1.0, score=0.486
============Step 3============
Evaluating 1294 rules


100%|██████████| 1294/1294 [00:01<00:00, 772.24it/s]
25882it [00:13, 1903.01it/s]                             


Fail: beam_bound=0.9304 / cause_bound=0.1157 / non_cause_bound=0.2804
Number of causes found: 2
Number of non-causes remaining: 1286
Best non-cause:
C={'FF-U5': '0'}, W={'FS-U1': '0', 'FN-U4': '1'}, output=0.9473684210526315, score=0.466
Worst non-cause:
C={'SD-U1': '0', 'DK': '0'}, W={'SD': '1'}, output=0.9473684210526315, score=0.483
============Step 4============
Evaluating 1273 rules


100%|██████████| 1273/1273 [00:01<00:00, 937.75it/s]
25461it [00:17, 1454.75it/s]                              

Fail: beam_bound=0.9377 / cause_bound=0.0000 / non_cause_bound=0.2856
Number of causes found: 0
Number of non-causes remaining: 1273
Best non-cause:
C={'SD': '0'}, W={'FS-U1': '0', 'FDB-U3': '1', 'FN-U1': '0'}, output=0.9473684210526315, score=0.481
Worst non-cause:
C={'DK': '0'}, W={'SD': '1', 'DK-U5': '0', 'GP-U5': '1'}, output=0.9047619047619048, score=0.481
----> Found 3 causes.
======Overall best rule:======
C={'SD': '0', 'DK': '0'}, W={}, output=0.0, score=0.448

RESULTS

C={'SD': '0', 'DK': '0'}, W={}, output=0.0, score=0.448
C={'A-U1': '0', 'DK': '0'}, W={'SD-U5': '0'}, output=0.16993464052287582, score=0.406
C={'SD-U1': '0', 'DK': '0'}, W={'SD-U5': '0'}, output=0.17791411042944785, score=0.439


In [72]:
avg_causes = [tuple(elt[3]) for elt in output_avg]
lucb_causes = [tuple(elt[3]) for elt in causes]
evaluate_full(avg_causes, ref_causes), evaluate_full(lucb_causes, ref_causes)

({'Accuracy': 0,
  'Recall': 0.03333333333333333,
  'Precision': 0.08333333333333333,
  'F1': 0.04761904761904761,
  'Missed': 0.6,
  '% Overshoot': 0.36666666666666664,
  'Average Overshoot': 1.5454545454545454,
  'Average Repeat': 1.0},
 {'Accuracy': 0,
  'Recall': 0.1,
  'Precision': 1.0,
  'F1': 0.18181818181818182,
  'Missed': 0.9,
  '% Overshoot': 0.0,
  'Average Overshoot': 0,
  'Average Repeat': 0})

In [44]:
set(avg_causes) & set(ref_causes)

{(20, 55), (25, 55), (40, 55), (50, 55), (56, 55)}

In [33]:
def cause_label(cause, variables):
    return sorted([variables[i] for i in cause])

In [46]:
for cause in ref_causes:
    print(f"{cause_label(cause, variables)}")
    print(f"   LUCB: {cause in lucb_causes}")
    print(f"   naive: {cause in avg_causes}")

['DK', 'SD']
   LUCB: True
   naive: True
['DK-U2', 'SD-U1']
   LUCB: False
   naive: False
['DK', 'SD-U1']
   LUCB: False
   naive: True
['DK-U2', 'SD']
   LUCB: False
   naive: False
['GP-U2', 'KMS-U1']
   LUCB: False
   naive: False
['GK-U2', 'KMS-U1']
   LUCB: False
   naive: False
['DK-U2', 'KMS-U1']
   LUCB: False
   naive: False
['GP-U2', 'SD-U1']
   LUCB: False
   naive: False
['GK-U2', 'SD-U1']
   LUCB: False
   naive: False
['DK', 'KMS-U1']
   LUCB: False
   naive: True
['GP-U2', 'SD']
   LUCB: False
   naive: False
['GK-U2', 'SD']
   LUCB: False
   naive: False
['A-U1', 'FN-U2', 'FS-U2']
   LUCB: False
   naive: False
['AD-U1', 'FN-U2', 'FS-U2']
   LUCB: False
   naive: False
['A-U1', 'GP-U2']
   LUCB: False
   naive: False
['AD-U1', 'GP-U2']
   LUCB: False
   naive: False
['FN-U2', 'FS-U2', 'KMS-U1']
   LUCB: False
   naive: False
['A-U1', 'FF-U2']
   LUCB: False
   naive: False
['AD-U1', 'FF-U2']
   LUCB: False
   naive: False
['A-U1', 'GK-U2']
   LUCB: False
   naive: Fal